In [ ]:
"""
ibge_coleta.py
==============
Coleta os endpoints IBGE de prioridade Alta definidos em:
  fontes_covariaveis_municipais.md

Fontes cobertas:
  1. IBGE Localidades   — /api/v1/localidades/municipios
  2. IBGE Censo 2022    — SIDRA tabelas 9606, 9605, 9514  (via apisidra)
"""

import time
import requests
import pandas as pd

# ─────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────

def get_json(url: str, params: dict = None, label: str = "") -> dict | list | None:
    """GET com tratamento de erro e log simples."""
    try:
        resp = requests.get(url, params=params, timeout=30)
        resp.raise_for_status()
        print(f"  ✔ {label or url}")
        return resp.json()
    except requests.HTTPError as e:
        print(f"  ✘ {label} — HTTP {e.response.status_code}")
    except Exception as e:
        print(f"  ✘ {label} — {e}")
    return None


# ══════════════════════════════════════════════
# 1. IBGE LOCALIDADES
# ══════════════════════════════════════════════

def coletar_localidades() -> pd.DataFrame:
    """
    Retorna todos os ~5.570 municípios com:
    id, nome, uf_sigla, uf_nome, regiao_imediata,
    regiao_intermediaria, macroregiao_sigla
    """
    print("\n[1/3] Coletando Localidades…")
    url = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"
    data = get_json(url, label="municipios")
    if not data:
        return pd.DataFrame()

    df = pd.DataFrame([
        {
            "id_municipio":         m["id"],
            "nome_municipio":       m["nome"],
            "uf_sigla":             m["regiao-imediata"]["regiao-intermediaria"]["UF"]["sigla"],
            "uf_nome":              m["regiao-imediata"]["regiao-intermediaria"]["UF"]["nome"],
            "regiao_imediata_id":   m["regiao-imediata"]["id"],
            "regiao_imediata_nome": m["regiao-imediata"]["nome"],
            "regiao_interm_id":     m["regiao-imediata"]["regiao-intermediaria"]["id"],
            "regiao_interm_nome":   m["regiao-imediata"]["regiao-intermediaria"]["nome"],
            "macroregiao_sigla":    m["regiao-imediata"]["regiao-intermediaria"]["UF"]["regiao"]["sigla"],
            "macroregiao_nome":     m["regiao-imediata"]["regiao-intermediaria"]["UF"]["regiao"]["nome"],
        }
        for m in data
    ])
    print(f"     → {len(df)} municípios carregados")
    return df


# ══════════════════════════════════════════════
# 2. IBGE CENSO 2022 — SIDRA
# ══════════════════════════════════════════════

# Tabelas definidas no documento:
#   9606 — Domicílios com internet por município
#   9605 — Rendimento médio domiciliar per capita
#   9514 — População por sexo e faixa etária
#
# Endpoint padrão apisidra:
#   /values/t/{tabela}/n6/all/v/all/p/last

SIDRA_BASE = "https://apisidra.ibge.gov.br/values"

TABELAS_CENSO = {
    "9606": "Domicilios_com_internet",
    "9605": "Rendimento_medio_domiciliar_percapita",
    "9514": "Populacao_sexo_faixa_etaria",
}


def _parse_sidra(data: list) -> pd.DataFrame:
    """
    A API SIDRA retorna uma lista onde o índice 0 é o cabeçalho
    e os demais são registros.
    """
    if not data or len(data) < 2:
        return pd.DataFrame()
    header = list(data[0].keys())
    rows   = data[1:]
    return pd.DataFrame(rows, columns=header)


def coletar_censo_sidra() -> dict[str, pd.DataFrame]:
    """
    Retorna um dict  {nome_tabela: DataFrame}
    para cada tabela do Censo 2022 listada acima.
    """
    print("\n[2/3] Coletando Censo 2022 via SIDRA…")
    resultados = {}

    for tabela, nome in TABELAS_CENSO.items():
        url = f"{SIDRA_BASE}/t/{tabela}/n6/all/v/all/p/last"
        data = get_json(url, label=f"SIDRA tabela {tabela} — {nome}")

        if data:
            df = _parse_sidra(data)
            print(f"     → {len(df)} linhas")
            resultados[nome] = df
        else:
            resultados[nome] = pd.DataFrame()

        time.sleep(0.5)   # respeita rate-limit da API

    return resultados


In [ ]:
coletar_localidades()


In [ ]:
coletar_censo_sidra()

In [ ]:
# ══════════════════════════════════════════════
# Main
# ══════════════════════════════════════════════

if __name__ == "__main__":

    # 1. Localidades ──────────────────────────
    df_localidades = coletar_localidades()
    print(df_localidades.head(3).to_string())

    # 2. Censo 2022 / SIDRA ───────────────────
    censo = coletar_censo_sidra()
    for nome, df in censo.items():
        print(f"\n── {nome} ({len(df)} linhas) ──")
        print(df.head(3).to_string())